In [ ]:
using Kinbiont
using CSV
using DataFrames
using Plots
using Statistics
using DecisionTree
using AbstractTrees
using MLJDecisionTreeInterface
using TreeRecipe
using Random


Data from ["Construction and Modeling of a Coculture Microplate for Real-Time Measurement of Microbial Interactions"](https://journals.asm.org/doi/10.1128/msystems.00017-21).

We consider the data for the Coculture of Gut drosphila microbiome. We obtained the annotation from the code of the github of the paper.

In [ ]:

# load data
data = CSV.read("../data/DATA_Gut_Microbiome_CoC.csv", DataFrame);
# convert time column from in elapsed time from the start in hours, the delta time in the data is 15 minutes

delta_time = 15/60;
time_seq = 0:delta_time:(size(data,1)-1)*delta_time;
time_seq = [ time_seq[i] for i in 1: length(time_seq)];
data[!,:Time] = time_seq ;

We initialize the model the parameter guess and the vector where store the results.

In [ ]:
model ="aHPM";
results_matrix = Kinbiont.initialize_df_results(model);

p_guess = [0.2, 0.001, 1.00, 1.0];
ub_ahpm = [3.0, 1.0,3.00, 5.0];
lb_ahpm = [0.001, 0.0,0.2, 0.0];

We start fitting the data of Acetobacter Oryzifermentans (AO). We consider the wells where is present such bacteria and in the connected wells there is only media. We make the average of the replicates and fit them.

In [ ]:
# Fitting all AO data
# Not interacting A0
# AO, -
# [1,21,41];
A0 = [1,21,41].+1;

data_A0 = data[:,A0] ;#.- blanS1_value

# mean of the replicates
data_A0_mean = mean(Matrix(data_A0),dims=2);

# Formatting data for Kinbiont
data_to_fit = permutedims( [time_seq data_A0_mean]);

# Performing ODE fitting
results_ODE_fit = Kinbiont.fitting_one_well_ODE_constrained(
    data_to_fit, 
    "AO_NA",
    "CoC_DT",
    model,
    p_guess;
    lb = lb_ahpm,
    ub = ub_ahpm,
);
Plots.scatter(data_to_fit[1,:],data_to_fit[2,:], xlabel="Time [h]", ylabel="OD [arb. units]", label=["Data AO" nothing],color=:black,markersize =1,marker=:cross ,size = (400,400))
display(Plots.plot!(results_ODE_fit[4],results_ODE_fit[3], xlabel="Time [h]", ylabel="OD [arb. units]",label=["fit AO" nothing],color=:grey,markersize =4 ,size = (400,400),linewidth=4))
results_matrix = [results_matrix results_ODE_fit[2]];


Now we select the data of AO where in the other wells are present the Lactobacillus Brevis.

In [ ]:
A0 = [13,33,53,17].+1;

data_A0 = data[:,A0]; #.- blanS1_value

# mean of the replicates
data_A0_mean = mean(Matrix(data_A0),dims=2);

# Formatting data for Kinbiont
data_to_fit = permutedims( [time_seq data_A0_mean]);

# Performing ODE fitting
results_ODE_fit = Kinbiont.fitting_one_well_ODE_constrained(
    data_to_fit, 
    "AO_LB",
    "CoC_DT",
    model,
    p_guess;
    lb = lb_ahpm,
    ub = ub_ahpm,
);
Plots.scatter!(data_to_fit[1,:],data_to_fit[2,:], xlabel="Time [h]", ylabel="OD [arb. units]", label=["Data AO+LB" nothing],color=:black,markersize =1,marker=:square ,size = (400,400))
display(Plots.plot!(results_ODE_fit[4],results_ODE_fit[3], xlabel="Time [h]", ylabel="OD [arb. units]",label=["fit AO+LB" nothing],color=:red,markersize =4 ,size = (400,400),linewidth=4,linestyle=:dot))


results_matrix = [results_matrix results_ODE_fit[2]];


We select the wells where AO interacts with L. Plantarum.

In [ ]:
A0 = [13,33,53,17].+1;

data_A0 = data[:,A0]; #.- blanS1_value

# mean of the replicates
data_A0_mean = mean(Matrix(data_A0),dims=2);

# Formatting data for Kinbiont
data_to_fit = permutedims( [time_seq data_A0_mean]);

# Performing ODE fitting
results_ODE_fit = Kinbiont.fitting_one_well_ODE_constrained(
    data_to_fit, 
    "AO_LB",
    "CoC_DT",
    model,
    p_guess;
    lb = lb_ahpm,
    ub = ub_ahpm,
);
Plots.scatter!(data_to_fit[1,:],data_to_fit[2,:], xlabel="Time [h]", ylabel="OD [arb. units]", label=["Data AO+LB" nothing],color=:black,markersize =1,marker=:square ,size = (400,400))
display(Plots.plot!(results_ODE_fit[4],results_ODE_fit[3], xlabel="Time [h]", ylabel="OD [arb. units]",label=["fit AO+LB" nothing],color=:red,markersize =4 ,size = (400,400),linewidth=4,linestyle=:dot)


results_matrix = [results_matrix results_ODE_fit[2]];


Finally the wells where are both present AO.

In [ ]:

# AO, AO
# [37,47,57];



A0 =[37,47,57].+1;

data_A0 = data[:,A0] ;#.- blanS1_value

# mean of the replicates
data_A0_mean = mean(Matrix(data_A0),dims=2);

# Formatting data for Kinbiont
data_to_fit = permutedims( [time_seq data_A0_mean]);

# Performing ODE fitting
results_ODE_fit = Kinbiont.fitting_one_well_ODE_constrained(
    data_to_fit, 
    "AO_AO",
    "CoC_DT",
    model,
    p_guess;
    lb = lb_ahpm,
    ub = ub_ahpm,
);
Plots.scatter!(data_to_fit[1,:],data_to_fit[2,:], xlabel="Time [h]", ylabel="OD [Arb. Units]", label=["Data AO+AO" nothing],color=:black,markersize =1 ,size = (300,300))
display(Plots.plot!(results_ODE_fit[4],results_ODE_fit[3], xlabel="Time [h]", ylabel="OD [Arb. Units]",label=["fit AO+AO" nothing],color=:blue,markersize =4 ,size = (600,500),legendposition = :topleft,linewidth=4,linestyle=:dashdot,tickfontsize = 20,labelfontsize = 20,legendfontsize =11))

results_matrix = [results_matrix results_ODE_fit[2]];


We stored the results in a matrix with this format ready for the next passage. 

In [ ]:
results_matrix

We create a feature matrix. For each curve we annotate the which strain where present in the other well. 

In [ ]:
# Creating feature Matrix


label_row = ["AO_NA","AO_LP","AO_LB","AO_AO"];
starting_0 = zeros(Int,4,3);
feature_matrix = hcat(label_row,starting_0);
feature_matrix[2,2] = 1 ;
feature_matrix[3,3] = 1 ;
feature_matrix[4,4] = 1 ;
feature_names = ["LP","LB","AO"];
feature_matrix

We set seed and we run a decision tree regression. Note that row to learn is 6. Meaning that we do a regression of the growth rate (the 6th row in result matrix).
After the regression we plot the tree.

In [ ]:
dt = Kinbiont.downstream_decision_tree_regression(results_matrix,
        feature_matrix,
        6;  # Row to learn
        do_pruning=false,
        verbose=true,
        do_cross_validation=false,
        max_depth=-1, 
        seed=seed,
        min_samples_leaf =1,
    #    min_purity_increase= 0.001, 
        min_samples_split=2
    )


In [ ]:

# Visualizing the decision tree

wt = DecisionTree.wrap(dt[1]);
p2 = Plots.plot(wt, 0.9, 0.2; size=(1400, 700), connect_labels=["yes", "no"]);
display(p2)

We repeated the analysis for the saturation value of the OD (N_max), i.e., the 4th row of the results matrix.

In [ ]:
dt = Kinbiont.downstream_decision_tree_regression(results_matrix,
        feature_matrix,
        4;  # Row to learn
        do_pruning=false,
        verbose=true,
        do_cross_validation=false,
        max_depth=depth, 
        seed=seed,
        min_samples_leaf =1,
     #   min_purity_increase= 0.0001, 
        min_samples_split=2
    )


In [ ]:

# Visualizing the decision tree

wt = DecisionTree.wrap(dt[1])
p2 = Plots.plot(wt, 0.9, 0.2; size=(1400, 700), connect_labels=["yes", "no"])